# Week 7B — Regularisation

## Applying regularisation to our molecular-properties dataset

This notebook has two parts:
1. **Decision-tree regularisation:** classify compounds into lower/higher Mannhold LogP groups and constrain tree complexity.
2. **Ridge and Lasso:** predict the continuous Mannhold LogP value from molecular descriptors and compare coefficient penalties.

The lower/higher LogP groups are created from the dataset median. They are not measured bioactivity labels.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42

## 2. Load our molecular-properties dataset

In [ ]:
df = pd.read_csv(r'C:\COLLEGE\CSC1171\Week 1\week3-DataIntro\notebooks\properties.csv')
df = df.replace("#NUM!", np.nan)

numeric_columns = [
    "Mannhold LogP", "Molecular Weight", "Hydrogen Bond Acceptors",
    "Hydrogen Bond Donors", "Topological Polar Surface Area",
    "Rotatable Bonds Count", "SP3 Character"
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.head()

## 3. Train/test split for LogP-group classification

In [ ]:
# Create a binary teaching target from the median Mannhold LogP.
# This is a LogP category, not a bioactivity category.
df_class = df.dropna(subset=["Mannhold LogP"]).copy()
logp_median = df_class["Mannhold LogP"].median()
df_class["logp_class"] = (df_class["Mannhold LogP"] >= logp_median).astype(int)

tree_features = ["Molecular Weight", "Topological Polar Surface Area"]
X = df_class[tree_features]
y = df_class["logp_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"Median Mannhold LogP: {logp_median:.3f}")
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 4. Regularising decision trees

A tree can be constrained with:
- `max_depth`: limits the number of levels.
- `min_samples_leaf`: requires a minimum number of compounds in each leaf.

These settings reduce the number of very specific rules. A simpler tree is not guaranteed to perform better on every test split.

In [ ]:
tree_settings = [
    ("Unrestricted", {}),
    ("Depth 3", {"max_depth": 3}),
    ("Depth 5, leaf size 5", {"max_depth": 5, "min_samples_leaf": 5}),
    ("Depth 3, leaf size 10", {"max_depth": 3, "min_samples_leaf": 10})
]

tree_rows = []
for name, settings in tree_settings:
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(random_state=RANDOM_STATE, **settings)
    )
    model.fit(X_train, y_train)
    tree_rows.append({
        "Model": name,
        "Leaves": model[-1].get_n_leaves(),
        "Training accuracy": model.score(X_train, y_train),
        "Test accuracy": model.score(X_test, y_test)
    })

pd.DataFrame(tree_rows).round(3)

**Look for:** the unrestricted tree may have more leaves and higher training accuracy. Does it also have the highest test accuracy? If not, that may indicate overfitting.

## 5. Ridge and Lasso: penalising coefficients

For this part, we predict the **continuous Mannhold LogP value** rather than the binary class.

We use molecular descriptors as predictors:
- Molecular Weight
- Hydrogen Bond Acceptors
- Hydrogen Bond Donors
- Topological Polar Surface Area
- Rotatable Bonds Count
- SP3 Character

We exclude `XLogP` because it is another LogP descriptor and could act as a close proxy for the target. The regression objective adds a penalty to the data-fitting loss:

**Objective = loss + alpha × penalty**

- **Ridge (L2):** penalises squared coefficients and shrinks them towards zero.
- **Lasso (L1):** penalises absolute coefficients and can set some coefficients exactly to zero (feature selection).

We use a continuous target here because Ridge and Lasso in this lesson are regression models.

In [ ]:
# Use rows with a valid target. Missing predictor values are imputed within each pipeline.
regression_features = [
    "Molecular Weight",
    "Hydrogen Bond Acceptors",
    "Hydrogen Bond Donors",
    "Topological Polar Surface Area",
    "Rotatable Bonds Count",
    "SP3 Character"
]
reg_df = df.dropna(subset=["Mannhold LogP"]).copy()
X_reg = reg_df[regression_features]
y_reg = reg_df["Mannhold LogP"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=RANDOM_STATE
)
print("Training compounds:", len(Xr_train), "| Test compounds:", len(Xr_test))
display(Xr_train.head())

## 6. Compare ordinary linear regression, Ridge, and Lasso

Features are **imputed and standardised inside each pipeline**. Standardisation matters because coefficient penalties depend on coefficient size, and coefficient size depends on feature units.

`alpha` controls penalty strength: larger values mean stronger regularisation. The three models are compared on the same held-out test set. Because Mannhold LogP contains extreme values, interpret the scores alongside the distribution and residuals.

In [ ]:
models = {
    "Linear regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression()),
    "Ridge (alpha=10)": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10)),
    "Lasso (alpha=0.1)": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Lasso(alpha=0.1, max_iter=20000))
}

score_rows, coefficient_rows = [], []
for name, model in models.items():
    model.fit(Xr_train, yr_train)
    pred_train = model.predict(Xr_train)
    pred_test = model.predict(Xr_test)
    score_rows.append({
        "Model": name,
        "Training R²": model.score(Xr_train, yr_train),
        "Test R²": model.score(Xr_test, yr_test),
        "Test MAE": mean_absolute_error(yr_test, pred_test),
        "Test RMSE": mean_squared_error(yr_test, pred_test) ** 0.5
    })
    regressor = model[-1]
    for feature, coef in zip(regression_features, regressor.coef_):
        coefficient_rows.append({"Model": name, "Feature": feature, "Coefficient": coef})

display(pd.DataFrame(score_rows).round(3))
coef_df = pd.DataFrame(coefficient_rows)
coef_pivot = coef_df.pivot(index="Feature", columns="Model", values="Coefficient")
coef_pivot = coef_pivot[list(models)]
display(coef_pivot.round(3))

In [ ]:
ax = coef_pivot.loc[regression_features[::-1]].plot(kind="barh", figsize=(10, 6))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Coefficient (standardised features)")
ax.set_ylabel("")
ax.set_title("Comparing Linear Regression, Ridge and Lasso coefficients")
plt.tight_layout()
plt.show()

**Look for:**
- The coefficients describe how each standardised feature contributes to the model's prediction, holding the other inputs constant.
- Ridge generally shrinks coefficients towards zero.
- Lasso may set weaker coefficients exactly to zero.
- Compare test R², MAE and RMSE rather than assuming the most complex or most regularised model is best.

Correlated molecular descriptors can make individual coefficients difficult to interpret. For example, molecular weight, TPSA and hydrogen-bond descriptors are related to one another.

## 7. Investigate the penalty strength

Compare how Ridge and Lasso coefficients change as `alpha` increases. The alpha ranges are illustrative and differ because Ridge and Lasso use differently scaled penalty objectives.

In [ ]:
ridge_alphas = np.logspace(-3, 4, 40)
lasso_alphas = np.logspace(-3, 1.5, 40)

ridge_path = [
    make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=a))
    .fit(Xr_train, yr_train)[-1].coef_
    for a in ridge_alphas
]
lasso_path = [
    make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Lasso(alpha=a, max_iter=30000))
    .fit(Xr_train, yr_train)[-1].coef_
    for a in lasso_alphas
]
ridge_path, lasso_path = np.asarray(ridge_path), np.asarray(lasso_path)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for j, feature in enumerate(regression_features):
    axes[0].plot(ridge_alphas, ridge_path[:, j], label=feature)
    axes[1].plot(lasso_alphas, lasso_path[:, j], label=feature)
for ax, title in zip(axes, ["Ridge (L2)", "Lasso (L1)"]):
    ax.set_xscale("log")
    ax.set_xlabel("alpha (stronger penalty →)")
    ax.set_title(title)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Coefficient (standardised features)")
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), title="Molecular descriptor")
plt.tight_layout()
plt.show()

**Look for:** Ridge coefficients usually approach zero smoothly but do not become exactly zero. Lasso can set coefficients to zero as the penalty increases. This illustrates how regularisation controls model complexity.

## Key takeaways

- Decision-tree regularisation (`max_depth`, `min_samples_leaf`) limits the complexity of classification rules.
- Ridge shrinks coefficients; Lasso can set some coefficients exactly to zero.
- Standardise features before coefficient penalties.
- Larger `alpha` means a stronger penalty.
- Use test metrics to compare models, and use cross-validation to select settings properly.

**Project limitation:** the classification section predicts LogP groups, and the regression section predicts Mannhold LogP. Neither section predicts bioactivity because this dataset workflow does not define a measured bioactivity target.